In [ ]:
# Install OS-level packages needed by unstructured (PDF parsing, OCR, file-type detection)
# for linux
# !apt-get install poppler-utils tesseract-ocr libmagic-dev

# for mac
# !brew install poppler tesseract libmagic

In [ ]:
# Install Python packages: unstructured (parsing), langchain + chroma (RAG/vector store), groq (LLM), huggingface (embeddings), dotenv (env vars)
%pip install -Uq "unstructured[all-docs]"
%pip install -Uq langchain_chroma
%pip install -Uq langchain langchain-community langchain-groq langchain-huggingface
%pip install -Uq sentence-transformers
%pip install -Uq python_dotenv

: 

In [ ]:
# Imports + load API keys from .env - shared pipeline logic lives in rag_pipeline.py
from dotenv import load_dotenv

import rag_pipeline

load_dotenv()

llm = rag_pipeline.get_llm()
embedding_model = rag_pipeline.get_embedding_model()

In [ ]:
# Step 1: Partition PDF into atomic elements (text, tables, images) - uses rag_pipeline.partition_document
file_path = "./docs/attention-is-all-you-need.pdf"  # Change this to your PDF path
elements = rag_pipeline.partition_document(file_path)
print(f"✅ Extracted {len(elements)} elements")

In [ ]:
# Inspect the raw extracted elements
# elements
# len(elements)

elements

In [ ]:
# All types of different atomic elements we see from unstructured
set([str(type(el)) for el in elements])

In [ ]:
# Peek at one element's full metadata/dict
elements[36].to_dict()

In [ ]:
# Gather all images
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()

# Use https://codebeautify.org/base64-to-image-converter to view the base64 text

In [ ]:
# Gather all table
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

# Use https://jsfiddle.net/ to view the table html

In [ ]:
# Step 2: Group atomic elements into chunks by title - uses rag_pipeline.create_chunks_by_title
chunks = rag_pipeline.create_chunks_by_title(elements)
print(f"✅ Created {len(chunks)} chunks")

In [ ]:
# View all chunks
# chunks

# All unique types
set([str(type(chunk)) for chunk in chunks])

In [ ]:
# View a single chunk
# chunks[2].to_dict()

# View original elements
chunks[11].metadata.orig_elements[-1].to_dict()
# Note: 4th chunk has the first image + 11th chunk has the first table in the sample PDF

In [ ]:
# Step 3: Summarize each chunk (AI summary for tables/images) and wrap into LangChain Documents - uses rag_pipeline.summarise_chunks
processed_chunks = rag_pipeline.summarise_chunks(llm, chunks)
print(f"✅ Processed {len(processed_chunks)} chunks")

In [ ]:
# Inspect the resulting LangChain Documents
processed_chunks

In [ ]:
# Helper: dump processed chunks/documents to a JSON file for inspection
import json

def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []

    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)

    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

In [ ]:
# Step 4: Embed documents and store them in a persisted ChromaDB vector store - uses rag_pipeline.build_vector_store
db = rag_pipeline.build_vector_store(processed_chunks, embedding_model, persist_directory="dbv1/chroma_db")
print("✅ Vector store created and saved to dbv1/chroma_db")

In [ ]:
# Quick test retrieval against the vector store just created, export results to JSON
query = "What are the two main components of the Transformer architecture? "
chunks = rag_pipeline.retrieve_chunks(db, query, k=3)

# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

In [ ]:
# Full pipeline: partition -> chunk -> summarize -> embed & store, in one call - uses rag_pipeline.run_ingestion_pipeline
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)

    db, documents = rag_pipeline.run_ingestion_pipeline(
        pdf_path, llm, embedding_model, persist_directory="dbv2/chroma_db"
    )

    print("🎉 Pipeline completed successfully!")
    return db

# Run the complete pipeline

In [ ]:
# Actually run the full ingestion pipeline on the target PDF
db = run_complete_ingestion_pipeline("./docs/attention-is-all-you-need.pdf")